# TRM (Tiny Recursive Model) no Colab — ARC-AGI

**7M params, ~45% ARC-AGI-1.** Roda o codigo OFICIAL (Samsung SAIL) num T4 gratis, downscaled p/ caber. Treino na nuvem; inferencia depois roda ate na RTX 3050.

**Por que e diferente:** nao e busca nem NCA single-shot — e RECURSAO com supervisao profunda: mantem (z=raciocinio, y=resposta) e refina . Profundidade-por-recursao da compute efetivo sem params p/ overfitar.

> **Runtime > Change runtime type > T4 GPU** antes de rodar.


## 1) Setup — clona o repo oficial + dados ARC


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone https://github.com/SamsungSAILMontreal/TinyRecursiveModels.git
%cd TinyRecursiveModels
!pip install -q -r requirements.txt 2>/dev/null || pip install -q torch einops tqdm omegaconf hydra-core wandb
!git clone https://github.com/fchollet/ARC-AGI.git /content/ARC-AGI
print("setup OK — leia o README do repo p/ os nomes exatos dos scripts/configs")
!ls


## 2) Dados + augmentacoes (d8 + permutacao de cor)

O TRM treina com muitas augmentacoes por tarefa. Use o script de dados do repo se houver ( / ver README); abaixo um gerador minimo de fallback.


In [ ]:
import os, json, glob, numpy as np
SRC="/content/ARC-AGI/data/training"; OUT="/content/arc_aug"; os.makedirs(OUT,exist_ok=True)
def d8(a): return [a,np.rot90(a,1),np.rot90(a,2),np.rot90(a,3),a[:,::-1],a[::-1,:],a.T,np.rot90(a.T,2)]
def aug_task(task,n=200):
    out=[]
    for _ in range(n):
        k=np.random.randint(8); lut=np.concatenate([[0],np.random.permutation(9)+1])
        f=lambda g: lut[d8(np.array(g))[k]].tolist()
        out.append({"train":[{"input":f(p["input"]),"output":f(p["output"])} for p in task["train"]],
                    "test":[{"input":f(p["input"]),"output":f(p["output"])} for p in task["test"]]})
    return out
files=glob.glob(SRC+"/*.json"); print(len(files),"tarefas base")
for fp in files:
    t=json.load(open(fp)); base=os.path.basename(fp)[:-5]
    for i,a in enumerate(aug_task(t)): json.dump(a,open(f"{OUT}/{base}_{i}.json","w"))
print("augmentacoes em",OUT, "->", len(glob.glob(OUT+"/*.json")))


## 3) Treino DOWNSCALED p/ T4

O repo treina com 4xH100. No T4: , ,  com gradient-accumulation, menos augmentacoes. **Ajuste os nomes dos args ao README/configs do repo** (Hydra). Salve checkpoints no Drive.


In [ ]:
# AJUSTE aos nomes reais do repo (veja  ou os configs em config/):
!python pretrain.py arch=trm data_path=/content/arc_aug \n  +arch.hidden_size=256 +arch.num_layers=2 \n  +training.global_batch_size=64 +training.gradient_accumulation_steps=8 \n  2>&1 | tail -50


## 4) Inferencia (roda ate na 3050 depois)

Most-common da predicao sobre as augmentacoes (inverter a transformacao antes de votar). Use o eval do repo apontando p/ . Pass@2 esperado: ~30-45% (downscaled fica abaixo do 45% full).


In [ ]:
!python evaluate.py checkpoint=CHECKPOINT_PATH data_path=/content/ARC-AGI/data/evaluation 2>&1 | tail -30


---
**Notas honestas:** (1) os nomes exatos de scripts/args do repo podem mudar — confira o README. (2) Downscaled no T4 fica abaixo do 45% full (treino oficial = 4xH100x3d); mede o teto na sua restricao. (3) Salve o checkpoint no Google Drive p/ rodar a inferencia localmente na 3050.
